In [ ]:
import numpy as np
from numpy.linalg import norm

import matplotlib.pyplot as plt

from scipy.sparse.linalg import eigsh

from functools import partial
from pathlib import Path

In [ ]:
from gpe_algorithms import inverse_iteration, dampened_inverse_iteration, shifted_inverse_iteration, calculate_energy
from gpe_problem import get_grid, get_L, get_L_M, get_M, A_v

# Plots

## Parameters

In [ ]:
max_steps = 200
tol = 1e-8
bound = 8.
steps = 200
betas = np.array([1e-1, 1e0, 1e+1, 1e+2, 1e+3])

## Precomputations

In [ ]:
# steps interior points, boundary points excluded (Dirichlet)
_, h = get_grid(bound, steps)
h_inv_sq = 1 / (h * h)
L_M = get_L_M(bound, steps, h_inv_sq)

L = get_L(steps, h_inv_sq)
V = get_M(bound, steps).diagonal()

## Helper methods for visualization

In [ ]:
def plot_result(axes, n_steps, dist_to_last, energy, diffs, color, label, success=True):
    """
    Plot the convergence history of one iteration on a 2x2 axes grid.

    The x-axis is the iteration index n = 1, ..., N of the iterate uⁿ
    (N = n_steps). The start vector u⁰ cannot be shown on the logarithmic
    x-axis, but E(u⁰) enters the energy difference at n = 1.

    Parameters
    ----------
    axes : ndarray of matplotlib.axes.Axes
        2x2 array of axes for plotting.
    n_steps : int
        Number of steps N taken in the iteration.
    dist_to_last : ndarray
        ||uⁿ - uᴺ|| for n = 1, ..., N-1.
    energy : ndarray
        E(uⁿ) for n = 0, ..., N.
    diffs : ndarray
        ||uⁿ - uⁿ⁻¹|| for n = 1, ..., N.
    color : str
        Color for the plots.
    label : str
        Label for the legend.
    success : bool, optional
        Whether the iteration converged. If not, the distance to the last iterate is not plotted
        (the last iterate is no approximation of the solution) and the lines
        are dashed.
    """
    n = np.arange(1, n_steps + 1)
    style = dict(c=color, marker='o', markersize=3, linestyle='-' if success else '--')

    if success:
        axes[0,0].plot(n[:-1], dist_to_last, **style)
    else:
        label = f"{label} (not converged)"

    axes[0,1].plot(n, diffs, label=label, **style)
    axes[1,0].plot(n, energy[1:], **style)
    axes[1,1].plot(n, energy[:-1] - energy[1:], **style)


def config_convergence_axes(ax_dist, ax_diff):
    """
    Configure the distance-to-last-iterate and differences axes (log-log).
    """
    for ax in (ax_dist, ax_diff):
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_xlabel('Iteration $n$')
    ax_dist.set_ylabel(r'Distance to last iterate $\Vert u^n - u^N \Vert$')
    ax_diff.set_ylabel(r'Differences $\Vert u^n - u^{n-1} \Vert$')


def config_energy_axes(ax_E, ax_dE):
    """
    Configure the energy and energy difference axes (log-linear).
    """
    for ax in (ax_E, ax_dE):
        ax.set_xscale('log')
        ax.set_yscale('linear')
        ax.set_xlabel('Iteration $n$')
    ax_E.set_ylabel('Energy $E(u^n)$')
    ax_dE.set_ylabel('Energy diff $E(u^{n-1}) - E(u^n)$')
    ax_dE.axhline(0, color='gray', linestyle='-.')


def config_result_plot(axes):
    """
    Configure a 2x2 axes grid for plotting iteration results.

    Parameters
    ----------
    axes : ndarray of matplotlib.axes.Axes
        2x2 array of axes to configure.
    """
    config_convergence_axes(axes[0,0], axes[0,1])
    config_energy_axes(axes[1,0], axes[1,1])


def match_limits(axes, which):
    """
    Give all axes the same x- or y-limits, computed from the plotted data.

    The limits span the data of all lines in all given axes with a margin
    of 5% (in log space for logarithmic axes). Reference lines that are not
    in data coordinates (e.g. axhline) are ignored. Must be called after the
    axis scales are set.

    Parameters
    ----------
    axes : iterable of matplotlib.axes.Axes
        Axes that should share the same limits.
    which : {'x', 'y'}
        Which axis limits to match.
    """
    axes = list(axes)
    values = [
        np.asarray(line.get_xdata() if which == 'x' else line.get_ydata(), dtype=float)
        for ax in axes for line in ax.get_lines()
        if line.get_transform() is ax.transData
    ]
    if not values:
        return
    values = np.concatenate(values)
    values = values[np.isfinite(values)]

    scale = axes[0].get_xscale() if which == 'x' else axes[0].get_yscale()
    if scale == 'log':
        values = values[values > 0]
    if values.size == 0:
        return

    lo, hi = values.min(), values.max()
    if scale == 'log':
        pad = (hi / lo) ** 0.05 if hi > lo else 2.
        lo, hi = lo / pad, hi * pad
    else:
        pad = 0.05 * (hi - lo) if hi > lo else max(0.5 * abs(hi), 1.)
        lo, hi = lo - pad, hi + pad

    for ax in axes:
        (ax.set_xlim if which == 'x' else ax.set_ylim)(lo, hi)

def add_legend(fig, ncol=5):
    """
    Add one legend for the whole figure below the axes, so it does not
    cover any data, and lay out the axes in the remaining space.
    """
    handles, labels = [], []
    for ax in fig.axes:
        h, l = ax.get_legend_handles_labels()
        handles += h
        labels += l
    n_rows = -(-len(labels) // ncol)
    fig.legend(handles, labels, loc='lower center', ncol=ncol, frameon=False)
    fig.tight_layout(rect=(0, 0.035 * n_rows, 1, 1))

img_dir="../img"
Path(img_dir).mkdir(parents=True, exist_ok=True)

## Helper function to run the inverse iteration algorithms and remove duplicate code

In [ ]:
def run_algo(algo, file_name):
    """
    Run an inverse iteration algorithm (normal, dampened or shifted) for multiple beta values,
    and plot the results on a 2x2 grid, saving the figure to `file_name`.

    Parameters
    ----------
    algo : function
        Iteration algorithm to run. Must accept the following parameters:
        - A : lambda taking vector v and returning a sparse matrix
        - calc_E : lambda taking vector v and returning energy
        - dim : int, problem dimension
        - max_steps : int, maximum iterations
        - tol : float, convergence tolerance
        Further parameters (e.g. the shift σ) can be bound with functools.partial.
    file_name : str
        Path to save the figure.

    """
    colors = ["darkgreen", "gold", "orange", "red", "darkred"]

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    for i, beta in enumerate(betas):
        # Define matrix operator and energy functional for this beta
        A = lambda v: A_v(L_M, v, beta, h_inv_sq)
        calc_E = lambda v: calculate_energy(v, L, V, beta, h_inv_sq)

        print(f"Starting iteration with bound={bound}, steps={steps}, beta={beta}")
        
        success, n_steps, last, dist_to_last, energy, diffs = algo(
            A, calc_E, steps * steps, max_steps, tol
        )

        if success:
            eigenval = last.T @ A(last) @ last / (last.T @ last)
            print(f"Successful iteration: converged in {n_steps} steps "
                  f"to eigenvalue ≈ {eigenval:.4f}")
        else:
            print(f"Iteration failed after {max_steps} steps for beta={beta}!")
        plot_result(axes, n_steps, dist_to_last, energy, diffs, colors[i], f"beta={beta}", success)

    config_result_plot(axes)
    add_legend(fig)
    plt.savefig(f"{img_dir}/{file_name}", dpi=300)
    plt.show()

## Testing

In [ ]:
run_algo(inverse_iteration, 'gpe-inverse-iteration.png')

In [ ]:
run_algo(dampened_inverse_iteration, 'gpe-inverse-iteration-dampened.png')

In [ ]:
# Fixed shift σ = θ·λ₀ [Hen24, Def. 5.1], where λ₀ is the smallest eigenvalue of the
# linear operator L_M (β = 0), a lower bound for the ground state eigenvalue of every β.
lambda_0 = eigsh(L_M, k=1, sigma=0, which='LM')[0][0]
theta = 0.9
sigma = theta * lambda_0
print(f"Smallest eigenvalue of the linear operator: lambda_0 = {lambda_0:.4f}, "
      f"shift sigma = {theta} * lambda_0 = {sigma:.4f}")

run_algo(partial(shifted_inverse_iteration, sigma=sigma), 'gpe-inverse-iteration-shifted.png')

## Comparision dampened - undampened

In [ ]:
d_colors = ["darkgreen", "forestgreen", "green", "limegreen", "springgreen"]
u_colors = ["gold", "orange", "darkorange", "red", "darkred"]

fig, axes = plt.subplots(3, 2, figsize=(12, 12))

# Rows 0 and 1: distance to last iterate and differences of the normal (u) and dampened (d)
# iteration. Row 2: energy and energy difference of both iterations.
u_axes = axes[[0,2], :]
d_axes = axes[[1,2], :]

for i, beta in enumerate(betas):
    A = lambda v: A_v(L_M, v, beta, h_inv_sq)
    calc_E = lambda v: calculate_energy(v, L, V, beta, h_inv_sq)

    print(f"Starting with bound={bound}, steps={steps}, beta={beta}")

    u_success, u_n_steps, u_last, u_dist_to_last, u_energy, u_diffs = inverse_iteration(A, calc_E, steps**2, max_steps, tol)
    d_success, d_n_steps, d_last, d_dist_to_last, d_energy, d_diffs = dampened_inverse_iteration(A, calc_E, steps**2, max_steps, tol)

    print(f"Normal iteration: converged={u_success} after {u_n_steps} steps; "
          f"dampened: converged={d_success} after {d_n_steps} steps")
    if u_success and d_success:
        print(f"Norm difference between results: {norm(u_last - d_last):.4e}")

    plot_result(u_axes, u_n_steps, u_dist_to_last, u_energy, u_diffs, u_colors[i], f"beta={beta} (normal)", u_success)
    plot_result(d_axes, d_n_steps, d_dist_to_last, d_energy, d_diffs, d_colors[i], f"beta={beta} (dampened)", d_success)

config_convergence_axes(axes[0,0], axes[0,1])
config_convergence_axes(axes[1,0], axes[1,1])
config_energy_axes(axes[2,0], axes[2,1])
axes[0,0].set_title('Inverse iteration')
axes[0,1].set_title('Inverse iteration')
axes[1,0].set_title('Dampened inverse iteration')
axes[1,1].set_title('Dampened inverse iteration')

# Same scales for normal and dampened iteration: one x-range for all
# panels, and one y-range for each quantity
match_limits(axes.flat, 'x')
match_limits(axes[:2, 0], 'y')
match_limits(axes[:2, 1], 'y')

add_legend(fig)
plt.savefig(f'{img_dir}/gpe-inverse-iteration-comparision.png', dpi=300)
plt.show()